In [103]:
from importlib.metadata import version
print("torch version:", version("torch"))

torch version: 2.2.2


In [104]:
import torch

inputs = torch.tensor(
	[[0.43, 0.15, 0.89], # Your     (x^1)
	[0.55, 0.87, 0.66], # journey  (x^2)
	[0.57, 0.85, 0.64], # starts   (x^3)
	[0.22, 0.58, 0.33], # with     (x^4)
	[0.77, 0.25, 0.10], # one      (x^5)
	[0.05, 0.80, 0.55]] # step     (x^6)
)

query = inputs[1]  # 2nd input token is the query

attn_scores_2 = torch.empty(inputs.shape[0])
for i , x_i in enumerate(inputs):
	attn_scores_2[i] = torch.dot(x_i,query)
print(inputs.shape[0])
print(attn_scores_2)
print(attn_scores_2.sum())
#Alternative way to get inner product
#print(inputs[0][0])
#res = [0]*(inputs.shape[0])
#print(res)
#for i in range(inputs.shape[0]):
#	for idx, element in enumerate(query):
#		res[i] += ((inputs[i][idx]).item())*(element.item())
#print(res)
#attn_scores_2_anotherway = torch.tensor(res)
#
#print(attn_scores_2_anotherway)


attn_weights_2_tmp = attn_scores_2 / attn_scores_2.sum()

print("Attention weights:", attn_weights_2_tmp)
print("Sum:", attn_weights_2_tmp.sum())

print(attn_scores_2.shape)

def softmax_naive(x):
	return torch.exp(x) / torch.exp(x).sum(dim = 0) #for (2,6) tensor, for example, use dim = 1 to sum each row separately

attn_weights_2_naive = softmax_naive(attn_scores_2)
print("Attention weights:", attn_weights_2_naive)
print("Sum:", attn_weights_2_naive.sum())

attn_weights_2 = torch.softmax(attn_scores_2, dim=0)
print("Attention weights:", attn_weights_2)
print("Sum:", attn_weights_2.sum())



6
tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])
tensor(6.5617)
Attention weights: tensor([0.1455, 0.2278, 0.2249, 0.1285, 0.1077, 0.1656])
Sum: tensor(1.0000)
torch.Size([6])
Attention weights: tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Sum: tensor(1.)
Attention weights: tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Sum: tensor(1.)


In [105]:
context_vec_2 = torch.zeros(query.shape)

print(context_vec_2)

for i, x_i in enumerate(inputs):
	context_vec_2 += attn_weights_2[i] * x_i
print(context_vec_2)



tensor([0., 0., 0.])
tensor([0.4419, 0.6515, 0.5683])


In [106]:
attn_scores = torch.empty(inputs.shape[0],inputs.shape[0])
print(attn_scores.shape)
for i,x_i in enumerate(inputs):
	for j,x_j in enumerate(inputs):
		attn_scores[i][j] = torch.dot(x_i,x_j)
print(attn_scores)


torch.Size([6, 6])
tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


In [107]:
attn_scores = inputs @ inputs.T
print(attn_scores)

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


In [108]:
attn_weights = torch.softmax(attn_scores, dim=-1)
print(attn_weights)

row_2_sum = sum(attn_weights[1].tolist())
print("Row 2 sum:", row_2_sum)

row_2_sum = attn_weights[1].sum()
print("Row 2 sum:", row_2_sum)

print("All row sums:", attn_weights.sum(dim=-1))

all_context_vecs  = attn_weights @ inputs
print(all_context_vecs)
print("Previous 2nd context vector:", context_vec_2)

tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])
Row 2 sum: 0.999999962747097
Row 2 sum: tensor(1.)
All row sums: tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000])
tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])
Previous 2nd context vector: tensor([0.4419, 0.6515, 0.5683])


In [109]:
x_2 = inputs[1]
print(inputs.shape)
d_in = inputs.shape[1]
d_out = 2

torch.manual_seed(123)
W_query = torch.nn.Parameter(torch.rand(d_in,d_out),requires_grad=False)
W_key   = torch.nn.Parameter(torch.rand(d_in,d_out),requires_grad=False)
W_value = torch.nn.Parameter(torch.rand(d_in,d_out),requires_grad=False)
print(W_query,W_key,W_value)
query_2 = x_2 @ W_query
key_2   = x_2 @ W_key
value_2 = x_2 @ W_value
print(query_2)

query  = inputs @ W_query
keys   = inputs @ W_key
values = inputs @ W_value
print(query.shape,keys.shape,values.shape)
keys_2 = keys[1]
attn_score_22 = torch.dot(keys_2,query_2) #or keys_2.dot(query_2) or keys_2 @ query_2 .T
print(attn_score_22)
attn_score_2  = query_2 @ keys.T
print(attn_score_2)

d_k = keys.shape[-1]
attn_weights_2 = torch.softmax(attn_score_2 / (d_k ** 0.5), dim=-1)
print(attn_weights_2)

context_vec_2 = attn_weights_2 @ values
print(context_vec_2)

torch.Size([6, 3])
Parameter containing:
tensor([[0.2961, 0.5166],
        [0.2517, 0.6886],
        [0.0740, 0.8665]]) Parameter containing:
tensor([[0.1366, 0.1025],
        [0.1841, 0.7264],
        [0.3153, 0.6871]]) Parameter containing:
tensor([[0.0756, 0.1966],
        [0.3164, 0.4017],
        [0.1186, 0.8274]])
tensor([0.4306, 1.4551])
torch.Size([6, 2]) torch.Size([6, 2]) torch.Size([6, 2])
tensor(1.8524)
tensor([1.2705, 1.8524, 1.8111, 1.0795, 0.5577, 1.5440])
tensor([0.1500, 0.2264, 0.2199, 0.1311, 0.0906, 0.1820])
tensor([0.3061, 0.8210])


In [110]:
import torch.nn as nn

class SelfAttention_v1(nn.Module):
	def __init__(self, d_in, d_out):
		super().__init__()
		self.W_query = nn.Parameter(torch.rand(d_in,d_out))
		self.W_key   = nn.Parameter(torch.rand(d_in,d_out))
		self.W_value = nn.Parameter(torch.rand(d_in,d_out))

	def forward(self,x):
		queries = x @ self.W_query
		keys    = x @ self.W_key
		values  = x @ self.W_value
		attn_scores = queries @ keys.T
		attn_weights = torch.softmax(
			attn_scores / (keys.shape[-1] ** 0.5), dim = -1
		) 
		context_vec = attn_weights @ values
		return context_vec

In [111]:
torch.manual_seed(123)
sa_v1 = SelfAttention_v1(d_in,d_out)
print(sa_v1(inputs))

tensor([[0.2996, 0.8053],
        [0.3061, 0.8210],
        [0.3058, 0.8203],
        [0.2948, 0.7939],
        [0.2927, 0.7891],
        [0.2990, 0.8040]], grad_fn=<MmBackward0>)


In [112]:
import torch.nn as nn

class SelfAttention_v2(nn.Module):
	def __init__(self, d_in, d_out,qkv_bias = False):
		super().__init__()
		self.W_query = nn.Linear(d_in,d_out,bias=qkv_bias)
		self.W_key   = nn.Linear(d_in,d_out,bias=qkv_bias)
		self.W_value = nn.Linear(d_in,d_out,bias=qkv_bias)

	def forward(self,x):
		queries = self.W_query(x)
		keys    = self.W_key(x)
		values  = self.W_value(x)
		attn_scores = queries @ keys.T
		attn_weights = torch.softmax(
			attn_scores / (keys.shape[-1] ** 0.5), dim = -1
		) 
		context_vec = attn_weights @ values
		return context_vec

In [113]:
torch.manual_seed(789)
sa_v2 = SelfAttention_v2(d_in,d_out)
print(sa_v2(inputs))

tensor([[-0.0739,  0.0713],
        [-0.0748,  0.0703],
        [-0.0749,  0.0702],
        [-0.0760,  0.0685],
        [-0.0763,  0.0679],
        [-0.0754,  0.0693]], grad_fn=<MmBackward0>)


In [114]:
queries = sa_v2.W_query(inputs)
keys = sa_v2.W_key(inputs)
attn_scores = queries @ keys.T
attn_weights = torch.softmax(
	attn_scores / (keys.shape[-1] ** 0.5), dim = -1
)
print(attn_weights)

tensor([[0.1921, 0.1646, 0.1652, 0.1550, 0.1721, 0.1510],
        [0.2041, 0.1659, 0.1662, 0.1496, 0.1665, 0.1477],
        [0.2036, 0.1659, 0.1662, 0.1498, 0.1664, 0.1480],
        [0.1869, 0.1667, 0.1668, 0.1571, 0.1661, 0.1564],
        [0.1830, 0.1669, 0.1670, 0.1588, 0.1658, 0.1585],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<SoftmaxBackward0>)


In [115]:
context_length = attn_weights.shape[0]
mask_simple = torch.tril(torch.ones(context_length,context_length))
print(mask_simple)

masked_simple = attn_weights * mask_simple
print(masked_simple)

row_sums = masked_simple.sum(dim=-1,keepdim = True)
print(row_sums)
masked_simple_norm = masked_simple / row_sums
print(masked_simple_norm)

tensor([[1., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1.]])
tensor([[0.1921, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2041, 0.1659, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2036, 0.1659, 0.1662, 0.0000, 0.0000, 0.0000],
        [0.1869, 0.1667, 0.1668, 0.1571, 0.0000, 0.0000],
        [0.1830, 0.1669, 0.1670, 0.1588, 0.1658, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<MulBackward0>)
tensor([[0.1921],
        [0.3700],
        [0.5357],
        [0.6775],
        [0.8415],
        [1.0000]], grad_fn=<SumBackward1>)
tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5517, 0.4483, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3800, 0.3097, 0.3103, 0.0000, 0.0000, 0.0000],
        [0.2758, 0.2460, 0.2462, 0.2319, 0.0000, 0.0000],
        [0.2175, 0.1983, 0.1984, 0.1888, 0.1971, 0

In [116]:
mask = torch.triu(torch.ones(context_length,context_length),diagonal = 1)
print(mask.bool())
masked = attn_scores.masked_fill(mask.bool(),-torch.inf)
print(masked)
attn_weights = torch.softmax(masked / keys.shape[-1]**0.5 , dim = -1)
print(attn_weights)

tensor([[False,  True,  True,  True,  True,  True],
        [False, False,  True,  True,  True,  True],
        [False, False, False,  True,  True,  True],
        [False, False, False, False,  True,  True],
        [False, False, False, False, False,  True],
        [False, False, False, False, False, False]])
tensor([[0.2899,   -inf,   -inf,   -inf,   -inf,   -inf],
        [0.4656, 0.1723,   -inf,   -inf,   -inf,   -inf],
        [0.4594, 0.1703, 0.1731,   -inf,   -inf,   -inf],
        [0.2642, 0.1024, 0.1036, 0.0186,   -inf,   -inf],
        [0.2183, 0.0874, 0.0882, 0.0177, 0.0786,   -inf],
        [0.3408, 0.1270, 0.1290, 0.0198, 0.1290, 0.0078]],
       grad_fn=<MaskedFillBackward0>)
tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5517, 0.4483, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3800, 0.3097, 0.3103, 0.0000, 0.0000, 0.0000],
        [0.2758, 0.2460, 0.2462, 0.2319, 0.0000, 0.0000],
        [0.2175, 0.1983, 0.1984, 0.1888, 0.1971, 0.0000],
        [0

In [117]:
torch.manual_seed(123)
dropout = torch.nn.Dropout(0.5)
example = torch.ones(6,6)
print(dropout(example))

tensor([[2., 2., 2., 2., 2., 2.],
        [0., 2., 0., 0., 0., 0.],
        [0., 0., 2., 0., 2., 0.],
        [2., 2., 0., 0., 0., 2.],
        [2., 0., 0., 0., 0., 2.],
        [0., 2., 0., 0., 0., 0.]])


In [118]:
torch.manual_seed(123)
print(dropout(attn_weights))

tensor([[2.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.8966, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.6206, 0.0000, 0.0000, 0.0000],
        [0.5517, 0.4921, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4350, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.3327, 0.0000, 0.0000, 0.0000, 0.0000]],
       grad_fn=<MulBackward0>)


In [119]:
batch = torch.stack((inputs,inputs),dim=0)
print(batch)     

tensor([[[0.4300, 0.1500, 0.8900],
         [0.5500, 0.8700, 0.6600],
         [0.5700, 0.8500, 0.6400],
         [0.2200, 0.5800, 0.3300],
         [0.7700, 0.2500, 0.1000],
         [0.0500, 0.8000, 0.5500]],

        [[0.4300, 0.1500, 0.8900],
         [0.5500, 0.8700, 0.6600],
         [0.5700, 0.8500, 0.6400],
         [0.2200, 0.5800, 0.3300],
         [0.7700, 0.2500, 0.1000],
         [0.0500, 0.8000, 0.5500]]])


In [120]:
import torch.nn as nn

class CausalAttention(nn.Module):
	def __init__(self, d_in, d_out, context_length, dropout, qkv_bias = False):
		super().__init__()
		self.W_query = nn.Linear(d_in,d_out,bias=qkv_bias)
		self.W_key   = nn.Linear(d_in,d_out,bias=qkv_bias)
		self.W_value = nn.Linear(d_in,d_out,bias=qkv_bias)
		self.dropout = nn.Dropout(dropout)
		self.register_buffer(
			'mask', torch.triu(torch.ones(context_length,context_length), diagonal=1)
		)
	
	def forward(self,x):
		b, num_tokens, d_in = x.shape
		queries = self.W_query(x)
		keys    = self.W_key(x)
		values  = self.W_value(x)

		attn_scores = queries @ keys.transpose(1,2)
		attn_scores.masked_fill_(
			self.mask.bool()[:num_tokens,:num_tokens], -torch.inf
		)
		attn_weights = torch.softmax(
			attn_scores / keys.shape[-1]**0.5 , dim = -1
		)
		attn_weights = self.dropout(attn_weights)

		context_vec = attn_weights @ values

		return context_vec
		

In [121]:
torch.manual_seed(123)
context_length = batch.shape[1]
ca = CausalAttention(d_in, d_out,context_length,0)
context_vecs = ca(batch)
print(context_vecs)

tensor([[[-0.4519,  0.2216],
         [-0.5874,  0.0058],
         [-0.6300, -0.0632],
         [-0.5675, -0.0843],
         [-0.5526, -0.0981],
         [-0.5299, -0.1081]],

        [[-0.4519,  0.2216],
         [-0.5874,  0.0058],
         [-0.6300, -0.0632],
         [-0.5675, -0.0843],
         [-0.5526, -0.0981],
         [-0.5299, -0.1081]]], grad_fn=<UnsafeViewBackward0>)


In [122]:
class MultiHeadAttentionWrapper(nn.Module):
	def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias = False):
		super().__init__()
		self.heads = nn.ModuleList(
			[CausalAttention(
				d_in,d_out,context_length,dropout,qkv_bias
				) 
				for _ in range(num_heads)]
		)
	
	def forward(self,x):
		return torch.cat([head(x) for head in self.heads], dim = -1)

In [123]:
torch.manual_seed(123)
context_length = batch.shape[1]
d_in, d_out = 3, 2
mha = MultiHeadAttentionWrapper(d_in,d_out,context_length,dropout=0,num_heads=2)
context_vecs = mha(batch)
print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)

tensor([[[-0.4519,  0.2216,  0.4772,  0.1063],
         [-0.5874,  0.0058,  0.5891,  0.3257],
         [-0.6300, -0.0632,  0.6202,  0.3860],
         [-0.5675, -0.0843,  0.5478,  0.3589],
         [-0.5526, -0.0981,  0.5321,  0.3428],
         [-0.5299, -0.1081,  0.5077,  0.3493]],

        [[-0.4519,  0.2216,  0.4772,  0.1063],
         [-0.5874,  0.0058,  0.5891,  0.3257],
         [-0.6300, -0.0632,  0.6202,  0.3860],
         [-0.5675, -0.0843,  0.5478,  0.3589],
         [-0.5526, -0.0981,  0.5321,  0.3428],
         [-0.5299, -0.1081,  0.5077,  0.3493]]], grad_fn=<CatBackward0>)
context_vecs.shape: torch.Size([2, 6, 4])


In [127]:
import torch.nn as nn

class MultiHeadAttention(nn.Module):
	def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
		super().__init__()
		assert (d_out % num_heads == 0), ("d_out must be divisible by num_heads")

		self.d_out = d_out
		self.num_heads = num_heads
		self.head_dim = d_out // num_heads
		self.W_query = nn.Linear(d_in,d_out,bias=qkv_bias)
		self.W_key = nn.Linear(d_in,d_out,bias=qkv_bias)
		self.W_value= nn.Linear(d_in,d_out,bias=qkv_bias)
		self.out_proj = nn.Linear(d_out,d_out)
		self.dropout = nn.Dropout(dropout)
		self.register_buffer(
			'mask', torch.triu(torch.ones(context_length,context_length),diagonal=1)
		)

	def forward(self, x):
		batch_size, num_tokens, d_in = x.shape
		queries = self.W_query(x)
		keys    = self.W_key(x)
		values  = self.W_value(x)

		queries = queries.view(batch_size,num_tokens,self.num_heads,self.head_dim)
		keys    = keys.view(batch_size,num_tokens,self.num_heads,self.head_dim)
		values  = values.view(batch_size,num_tokens,self.num_heads,self.head_dim)

		queries = queries.transpose(1,2)
		keys    = keys.transpose(1,2)
		values  = values.transpose(1,2)

		attn_scores = queries @ keys.transpose(2,3)
		mask_bool = self.mask.bool()[:num_tokens, :num_tokens]
		attn_scores.masked_fill_(mask_bool,-torch.inf)

		attn_weights = torch.softmax(
			attn_scores / (keys.shape[-1] ** (0.5)), dim = -1
		)
		attn_weights = self.dropout(attn_weights)

		context_vec = attn_weights @ values
		context_vec = context_vec.transpose(1,2)
		context_vec = context_vec.contiguous().view(batch_size,num_tokens,self.d_out)
		context_vec = self.out_proj(context_vec)

		return context_vec
		

In [128]:
torch.manual_seed(123)
batch_size, context_length, d_in = batch.shape
d_out = 2
print(batch.shape)
mha = MultiHeadAttention(d_in,d_out,context_length,dropout=0,num_heads=2)
context_vecs = mha(batch)
print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)

torch.Size([2, 6, 3])
tensor([[[0.3190, 0.4858],
         [0.2943, 0.3897],
         [0.2856, 0.3593],
         [0.2693, 0.3873],
         [0.2639, 0.3928],
         [0.2575, 0.4028]],

        [[0.3190, 0.4858],
         [0.2943, 0.3897],
         [0.2856, 0.3593],
         [0.2693, 0.3873],
         [0.2639, 0.3928],
         [0.2575, 0.4028]]], grad_fn=<ViewBackward0>)
context_vecs.shape: torch.Size([2, 6, 2])
